# 04 — Approximate Time-to-Churn (Survival Analysis)
### SmartTel Retention Intelligence — (Future Work / Experimental Extension)

**Input:** `telco_features_with_personas_latest.parquet` (produced by `03_classification_benchmarking.ipynb`)

> **Status: Future Work.** This module is scoped separately from the core system (Modules 1–3, 6–8), which is complete and deployable without it. It is included here because it extends the churn-risk view from a single point-in-time probability to a *time-aware* one, which is directly useful for scheduling retention outreach, but it rests on an approximation that is stated plainly below and should not be glossed over.

**Scope of this notebook**:
1. Construct the survival dataset (`duration = tenure`, `event = Churn`)
2. Kaplan-Meier curves, overall and stratified by Contract, PaymentMethod, and persona
3. Log-rank tests confirming stratified differences are statistically real
4. Cox Proportional Hazards model — hazard ratios per feature, with the proportional-hazards assumption explicitly checked (not assumed)
5. Random Survival Forest as a non-linear comparison, evaluated against Cox via concordance index (C-index)
6. Per-customer expected time-to-churn, to feed Module 6's CLV estimate
7. Persist outputs, with the approximation caveat carried into the column names and metadata themselves


## 0. What This Module Can and Cannot Claim

**What is legitimate here:** using `tenure` as the survival duration and `Churn` as the event indicator is a standard, textbook-correct setup for right-censored survival analysis, customers who haven't churned are censored at their current tenure; customers who have churned have an observed event time. Kaplan-Meier and Cox models fit to this structure are methodologically valid, not a misuse of the technique.

**What is *not* available, and why this is labeled an approximation:** this is a single cross-sectional extract of the customer base at one point in time, not a tracked cohort observed over calendar time. Concretely, that means:
- There is no way to validate these survival estimates against **actual future churn events** — everything here is fit and evaluated on the same static snapshot (via train/test splitting, which helps, but doesn't substitute for out-of-time validation)
- Hazard ratios reflect associations **within this snapshot's customer mix**; if the underlying customer base or market conditions shift, the fitted hazards may not transfer forward
- Any customer whose tenure is still accumulating at the moment of extraction is right-censored *at that instant* — there is no way to distinguish "about to churn next month" from "will stay for years" among currently-active customers beyond what the model infers from their covariates

These estimates are used **directionally** in this project to rank customers by urgency, and to feed a remaining-tenure estimate into CLV calculation — not as calendar-accurate forecasts of when any individual customer will leave.
